# Laboratório 4 — Calibração de Câmeras

**Disciplina:** ESZA019 — Visão Computacional — UFABC 2026.2
**Professor:** Celso Setsuo Kurashima

### Integrantes da equipe
- Lucas Rodrigues Teixeira — RA 11202131394
- Pedro Henrique Garcez Silva — RA 11202130642
- Roberto Sene Azevedo — RA 11202020360

**Data de realização dos experimentos:** 01/07/2026
**Data de publicação do relatório:** 08/07/2026


## Sumário

1. [Introdução](#1-introdução)
2. [Fundamentação teórica](#2-fundamentação-teórica)
   - 2.1 [Modelo de câmera *pinhole* e formação da imagem](#21-modelo-de-câmera-pinhole-e-formação-da-imagem)
   - 2.2 [Parâmetros intrínsecos — a matriz K](#22-parâmetros-intrínsecos--a-matriz-k)
   - 2.3 [Parâmetros extrínsecos — R e t](#23-parâmetros-extrínsecos--r-e-t)
   - 2.4 [Distorção das lentes — o vetor dist](#24-distorção-das-lentes--o-vetor-dist)
   - 2.5 [O processo de calibração com tabuleiro de xadrez](#25-o-processo-de-calibração-com-tabuleiro-de-xadrez)
3. [Ambiente e instruções de reprodução](#3-ambiente-e-instruções-de-reprodução)
4. [Procedimentos experimentais (Parte 2)](#4-procedimentos-experimentais-parte-2)
   - 4.1 [(A) Calibração com as imagens de exemplo fornecidas](#41-a-calibração-com-as-imagens-de-exemplo-fornecidas)
   - 4.2 [(B) Calibração da webcam da equipe](#42-b-calibração-da-webcam-da-equipe)
   - 4.3 [(C) Calibração de uma segunda câmera pessoal](#43-c-calibração-de-uma-segunda-câmera-pessoal)
   - 4.4 [(D) Correção da distorção das imagens](#44-d-correção-da-distorção-das-imagens)
5. [Análise e discussão](#5-análise-e-discussão)
6. [Declaração de uso de Inteligência Artificial Generativa](#6-declaração-de-uso-de-inteligência-artificial-generativa)
7. [Referências](#7-referências)


## 1. Introdução

Nos laboratórios anteriores a equipe estudou a captura de imagens/vídeos (Lab 1), a extração e
correspondência de **características** com SIFT (Lab 2) e o **alinhamento por homografia** e mosaico
(Lab 3). Todos esses experimentos trataram a imagem como um dado já formado. Neste Laboratório 4
recuamos um passo e estudamos **como a imagem é formada** dentro da câmera e como **medir**
numericamente as propriedades ópticas e geométricas desse processo — a **calibração de câmeras**.

Calibrar uma câmera significa estimar os **parâmetros intrínsecos** (distância focal, ponto principal,
distorção das lentes) e os **parâmetros extrínsecos** (posição e orientação da câmera em relação ao
mundo) a partir de imagens de um padrão de geometria conhecida, tipicamente um **tabuleiro de xadrez**.
Esses parâmetros são a base de praticamente toda a Visão Computacional métrica: retirar a distorção das
lentes, medir objetos, reconstruir cenas em 3D, visão estéreo (Lab 5) e SLAM.

Este relatório tem caráter **didático e autossuficiente**. Ele apresenta a teoria mínima necessária,
descreve e executa os experimentos dos itens (A) a (D) do roteiro, lista os parâmetros efetivamente
obtidos pela equipe (matriz $K$, vetor de distorção, vetores $R$ e $t$) e discute o significado de cada
um, de modo que outro aluno consiga reproduzir integralmente o experimento.


## 2. Fundamentação teórica

### 2.1 Modelo de câmera *pinhole* e formação da imagem

O modelo mais simples de câmera é o **pinhole** (câmera de furo): a luz de um ponto 3D do mundo
$M = (X, Y, Z)$ atravessa um único ponto (o centro óptico) e projeta-se no plano da imagem, gerando o
ponto 2D $m = (u, v)$. A relação, em **coordenadas homogêneas**, é

$$s \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = K\,[\,R \mid t\,] \begin{bmatrix} X \\ Y \\ Z \\ 1 \end{bmatrix}$$

onde $s$ é um fator de escala, $K$ é a matriz **intrínseca** (propriedades internas da câmera) e
$[R \mid t]$ são os **extrínsecos** (rotação $R$ e translação $t$ que levam do sistema de coordenadas do
mundo para o da câmera). A câmera real difere do pinhole ideal por causa das **lentes**, que introduzem
**distorção** — corrigida por um modelo polinomial adicional (o vetor `dist`).


### 2.2 Parâmetros intrínsecos — a matriz K

A matriz intrínseca tem a forma

$$K = \begin{bmatrix} f_x & s & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

- $f_x, f_y$ — **distância focal** expressa em *pixels* nas direções horizontal e vertical. Ela combina
  a distância focal física da lente com o tamanho do pixel do sensor.
- $c_x, c_y$ — **ponto principal**, isto é, onde o eixo óptico fura o plano da imagem (idealmente perto
  do centro da imagem).
- $s$ — **skew** (cisalhamento), que mede o quanto os eixos do pixel não são perpendiculares. Em
  sensores modernos é praticamente **zero**.
- **Razão de aspecto** (*aspect ratio*) = $f_y / f_x$; vale $\approx 1$ quando o pixel é quadrado.

$K$ é uma propriedade **da câmera** (com uma dada lente e foco): não muda quando a câmera se move.


### 2.3 Parâmetros extrínsecos — R e t

Enquanto $K$ descreve a câmera "por dentro", os **extrínsecos** descrevem **onde a câmera está** em
relação ao objeto observado:

- $R$ — matriz de **rotação** $3\times3$ (no OpenCV é retornada de forma compacta como um **vetor de
  Rodrigues** `rvec`, que pode ser convertido em matriz com `cv2.Rodrigues`).
- $t$ — vetor de **translação** $3\times1$ (`tvec`), a posição da origem do mundo vista pela câmera.

Como cada foto de calibração vê o tabuleiro em **uma pose diferente**, a calibração devolve **um par
$(R, t)$ para cada imagem** — cada par posiciona aquele tabuleiro específico em relação à câmera. Os
intrínsecos $K$ e `dist`, ao contrário, são **únicos** para todas as imagens (a câmera é a mesma).


### 2.4 Distorção das lentes — o vetor dist

Lentes reais curvam os raios de luz e fazem linhas retas parecerem curvas, sobretudo perto das bordas.
O OpenCV modela isso com o vetor

$$\texttt{dist} = (k_1,\; k_2,\; p_1,\; p_2,\; k_3)$$

- $k_1, k_2, k_3$ — coeficientes de **distorção radial** (o efeito "barril" ou "almofada"): pontos são
  deslocados radialmente em função da distância ao centro.
- $p_1, p_2$ — coeficientes de **distorção tangencial**, causada quando a lente não é perfeitamente
  paralela ao sensor.

Conhecendo `dist`, é possível **desfazer** a distorção (*undistortion*) e recuperar uma imagem em que as
retas do mundo voltam a ser retas — exatamente o que faremos no item (D).


### 2.5 O processo de calibração com tabuleiro de xadrez

O tabuleiro de xadrez é usado porque seus **cantos internos** formam uma grade **plana e regular**, de
geometria conhecida. O procedimento é:

1. Definir os pontos 3D do padrão (`objpoints`) — as coordenadas $(X, Y, 0)$ de cada canto, com $Z=0$
   porque o tabuleiro é plano.
2. Para cada foto, detectar os cantos na imagem (`cv2.findChessboardCorners`) e refiná-los ao nível de
   **subpixel** (`cv2.cornerSubPix`), formando os `imgpoints` (pontos 2D).
3. Resolver, com `cv2.calibrateCamera`, o conjunto de equações que relaciona `objpoints` e `imgpoints`,
   obtendo $K$, `dist` e os pares $(R, t)$ de cada imagem, além do **erro de reprojeção** (RMS, em
   pixels), que mede a qualidade da calibração — quanto menor, melhor.

**Atenção ao número de cantos internos.** O `CHECKERBOARD` deve ser o número de **cantos internos**
(interseções entre quadrados), não o número de quadrados. Neste laboratório usamos $(6, 8)$.


## 3. Ambiente e instruções de reprodução

Os experimentos foram executados em **Linux**, com **Python 3** e **OpenCV 4.x**. Para reproduzir:

```bash
# 1) Ambiente virtual
python3 -m venv venv
source venv/bin/activate

# 2) Dependências
pip install opencv-python numpy

# 3) (A) Calibração com as imagens de exemplo fornecidas (left*.jpg)
python3 L4_cal.py

# 4) (B/C) Capturar as próprias imagens do tabuleiro pela câmera (tecla 's' salva, 'q' sai)
python3 L4_chess.py
#     e depois calibrar essas imagens ajustando o glob e CHECKERBOARD=(6,8) em L4_cal.py
python3 L4_cal.py

# 5) (D) A correção de distorção está implementada nas células deste notebook.
```

> **Observação.** `L4_cal.py` lê as imagens por um padrão `glob` (por exemplo `left*.jpg` ou
> `roberto*.jpg`). Para calibrar um conjunto específico, ajuste esse padrão e o `CHECKERBOARD`. As
> imagens de entrada (`left*.jpg`, `roberto*.jpg`, `roberto_outra_camera*.jpg`) e os arquivos de
> parâmetros (`parametros_2a.txt`, `parametros_2b.txt`) estão nesta mesma pasta.


## 4. Procedimentos experimentais (Parte 2)

### 4.1 (A) Calibração com as imagens de exemplo fornecidas

**Objetivo:** executar o programa `L4_cal.py` sobre o conjunto de imagens de exemplo do tutorial
(`left01.jpg` … `left14.jpg`), detectar os cantos do tabuleiro e obter os parâmetros de calibração.

O programa abaixo é o `L4_cal.py` da equipe (comentado). Ele detecta os cantos em cada imagem, refina-os
em subpixel e chama `cv2.calibrateCamera`.


In [ ]:
import cv2
import numpy as np
import glob

# Dimensoes do tabuleiro = numero de CANTOS INTERNOS (nao de quadrados).
CHECKERBOARD = (6, 8)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Vetores de pontos 3D (mundo) e 2D (imagem) para cada foto.
objpoints = []   # pontos 3D do padrao (Z=0, tabuleiro plano)
imgpoints = []   # pontos 2D detectados na imagem

# Coordenadas do padrao no seu proprio sistema (uma grade regular de cantos).
objp = np.zeros((1, CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[0, :, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

# Entrada: imagens de exemplo fornecidas.
images = glob.glob('left*.jpg')
for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Procura os cantos internos do tabuleiro.
    ret, corners = cv2.findChessboardCorners(
        gray, CHECKERBOARD,
        cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_FAST_CHECK + cv2.CALIB_CB_NORMALIZE_IMAGE)

    # So usa a imagem se o tabuleiro inteiro foi encontrado.
    if ret:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)  # refino subpixel
        imgpoints.append(corners2)
        img = cv2.drawChessboardCorners(img, CHECKERBOARD, corners2, ret)

    cv2.imshow('img', img)
    cv2.waitKey(0)
cv2.destroyAllWindows()

# Calibracao: resolve K, dist e os pares (R, t) de cada imagem.
h, w = img.shape[:2]
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, gray.shape[::-1], None, None)

print("Camera matrix (K):\n", mtx)
print("dist:\n", dist)
print("rvecs (um por imagem):\n", rvecs)
print("tvecs (um por imagem):\n", tvecs)


**Parâmetros obtidos pela equipe (arquivo `parametros_2a.txt`).**

Matriz intrínseca $K$ (13 imagens de exemplo aproveitadas):

$$K_A = \begin{bmatrix} 536{,}07 & 0 & 342{,}37 \\ 0 & 536{,}02 & 235{,}54 \\ 0 & 0 & 1 \end{bmatrix}$$

Vetor de distorção (ordem $k_1, k_2, p_1, p_2, k_3$):

$$\texttt{dist}_A = [\,-0{,}2651,\; -0{,}0467,\; 0{,}00183,\; -0{,}000315,\; 0{,}2523\,]$$

A calibração também devolveu **13 pares $(R, t)$** — um `rvec` (rotação de Rodrigues) e um `tvec`
(translação) para cada imagem. Como exemplo, a primeira imagem tem
$rvec_1 = [-0{,}084,\; 0{,}348,\; -1{,}542]$ e $tvec_1 = [-2{,}962,\; 0{,}572,\; 16{,}830]$ (o tabuleiro
está a ~16,8 unidades de distância à frente da câmera). Os 13 pares completos estão em
`parametros_2a.txt`.

**Significado dos parâmetros (pergunta do roteiro).**
- **$K$** (intrínseca): distância focal em pixels ($f_x = 536{,}07$, $f_y = 536{,}02$) e ponto principal
  ($c_x = 342{,}37$, $c_y = 235{,}54$), próximo do centro de uma imagem $640\times480$. O **skew** é
  zero.
- **$R$** (via `rvec`): a orientação (rotação) do tabuleiro em relação à câmera **naquela** foto.
- **$t$** (`tvec`): a posição (translação) do tabuleiro em relação à câmera naquela foto.
- **`dist`**: os coeficientes de distorção radial ($k_1, k_2, k_3$) e tangencial ($p_1, p_2$) da lente.


### 4.2 (B) Calibração da webcam da equipe

**Objetivo:** capturar entre 10 e 15 imagens próprias do tabuleiro com a webcam (programa `L4_chess.py`,
alterando o nome do arquivo para o de um integrante — `roberto`) e, em seguida, calibrar a câmera com
`L4_cal.py` usando `CHECKERBOARD = (6, 8)`.

O programa de captura `L4_chess.py` (abaixo) exibe o vídeo ao vivo e grava o quadro atual, numerado, a
cada vez que a tecla **`s`** é pressionada (encerra com **`q`**).


In [ ]:
import numpy as np
import cv2 as cv

cap = cv.VideoCapture(0)
if not cap.isOpened():
    print("Cannot open camera")
    exit()

i = 0
while True:
    ret, frame = cap.read()            # captura um quadro da webcam
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break

    cv.imshow('frame', frame)          # mostra o vídeo ao vivo

    k = cv.waitKey(1)
    if k == ord('s'):                  # 's' salva o quadro atual, numerado
        cv.imwrite("roberto" + str(i) + ".jpg", frame)
        i = i + 1
        print("frame", i)
    elif k == ord('q'):                # 'q' encerra
        break

cap.release()
cv.destroyAllWindows()


**Parâmetros obtidos pela equipe (arquivo `parametros_2b.txt`).**

Matriz intrínseca $K$ da câmera da equipe:

$$K_B = \begin{bmatrix} 692{,}56 & 0 & 320{,}24 \\ 0 & 689{,}08 & 247{,}95 \\ 0 & 0 & 1 \end{bmatrix}$$

Vetor de distorção:

$$\texttt{dist}_B = [\,-0{,}0325,\; 0{,}4864,\; 0{,}00254,\; -0{,}00971,\; -1{,}2327\,]$$

Foram obtidos **23 pares $(R, t)$**, um para cada imagem de calibração em que o tabuleiro foi detectado.

**Valores pedidos no roteiro (a partir de $K_B$):**

| Parâmetro | Valor obtido |
|-----------|--------------|
| *Focal length* | $f_x = 692{,}56$ px, $f_y = 689{,}08$ px |
| *Aspect ratio* ($f_y/f_x$) | $0{,}9950$ (pixel praticamente quadrado) |
| *Skew* | $0$ (eixos ortogonais) |
| *Principal point* | $(320{,}24,\; 247{,}95)$ px — muito próximo do centro de $640\times480$ |

**Comentário.** A razão de aspecto $\approx 1$ e o *skew* nulo indicam pixels quadrados e sensor bem
alinhado, como esperado de uma webcam moderna. O ponto principal caiu quase no centro geométrico da
imagem, o que também é sinal de uma calibração saudável.

**Comparação com o item (A).** A diferença mais marcante é a **distância focal**: $\approx 692$ px na
câmera da equipe contra $\approx 536$ px nas imagens de exemplo. Como ambas as imagens são
$640\times480$, um $f$ maior significa um **campo de visão mais estreito** (mais "zoom") na câmera da
equipe. Os coeficientes de distorção também diferem: as imagens de exemplo têm $k_1 < 0$ dominante
(distorção tipo barril moderada), enquanto a câmera da equipe apresenta um $k_3$ negativo grande,
indicando um perfil de distorção radial diferente — coerente com lentes distintas.

**Por que resulta um par $(R, t)$ para cada imagem? (pergunta do roteiro).** Porque $R$ e $t$ são
**extrínsecos**: descrevem a pose (orientação e posição) **do tabuleiro em relação à câmera** em cada
foto, e o tabuleiro foi fotografado em **poses diferentes**. Cada par leva o sistema de coordenadas do
**padrão** (mundo) para o sistema de coordenadas da **câmera** daquela captura específica. Já os
intrínsecos $K$ e `dist` pertencem à câmera e são os **mesmos** para todas as imagens.


### 4.3 (C) Calibração de uma segunda câmera pessoal

**Objetivo:** repetir o procedimento do item (B) com uma **segunda câmera** e comparar os parâmetros.

A equipe capturou o conjunto de imagens `roberto_outra_camera0.jpg` … `roberto_outra_camera9.jpg`
(presentes nesta pasta), destinado à segunda câmera. Basta calibrá-lo apontando o `glob` de `L4_cal.py`
para esse conjunto:


In [ ]:
import cv2, numpy as np, glob

CHECKERBOARD = (6, 8)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
objp = np.zeros((1, CHECKERBOARD[0]*CHECKERBOARD[1], 3), np.float32)
objp[0, :, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

objpoints, imgpoints = [], []
gray = None
for fname in glob.glob('roberto_outra_camera*.jpg'):   # <-- segunda camera
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(
        gray, CHECKERBOARD,
        cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_FAST_CHECK + cv2.CALIB_CB_NORMALIZE_IMAGE)
    if ret:
        objpoints.append(objp)
        imgpoints.append(cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria))

ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)
print("K (segunda camera):\n", mtx)
print("dist:\n", dist.ravel())
print("Erro de reprojecao RMS:", round(ret, 4), "px")


> **Pendência a completar pela equipe.** Diferentemente dos itens (A) e (B), **não há um arquivo de
> parâmetros salvo** especificamente para esta segunda câmera na pasta — apenas as imagens de entrada
> (`roberto_outra_camera*.jpg`). A célula acima está pronta para gerar $K$, `dist` e o erro de
> reprojeção dessa câmera; a equipe deve **executá-la** e colar aqui a matriz $K$, o vetor `dist` e a
> comparação com o item (B) (distância focal, ponto principal, distorção). Recomenda-se ainda
> **confirmar** que as imagens `roberto_outra_camera*.jpg` foram de fato tiradas com uma câmera
> **diferente** da do item (B), pois um padrão `glob` como `roberto*.jpg` captura, sem querer,
> **os dois** conjuntos ao mesmo tempo — o que mistura câmeras distintas numa única calibração e deve
> ser evitado.


### 4.4 (D) Correção da distorção das imagens

**Objetivo:** usando a câmera já calibrada (parâmetros do item B), **corrigir a distorção** de imagens
próprias. O OpenCV oferece dois métodos; antes de ambos refina-se a matriz da câmera com
`cv2.getOptimalNewCameraMatrix`, que aceita um parâmetro de escala livre ($\alpha$):

1. `cv2.undistort()` — corrige a imagem inteira de uma vez.
2. *Remapping* com `cv2.initUndistortRectifyMap` + `cv2.remap()` — calcula mapas de correção e os aplica;
   é o método preferido quando se corrige **muitas** imagens (ou vídeo), pois os mapas são calculados uma
   única vez.

A célula abaixo é o programa de correção elaborado no notebook. Ele usa os parâmetros reais da câmera da
equipe (`parametros_2b.txt`) e uma imagem capturada por ela.


In [ ]:
import cv2, numpy as np

# Parametros REAIS da camera da equipe (parametros_2b.txt, item B).
K = np.array([[692.56010758, 0.,           320.24078754],
              [0.,           689.07613523, 247.94659559],
              [0.,           0.,           1.]])
dist = np.array([[-0.03251937, 0.48635413, 0.00253602, -0.00970879, -1.23269435]])

# Entrada: uma imagem capturada pela mesma camera.
img = cv2.imread('roberto3.jpg')
h, w = img.shape[:2]

# Refina a matriz da camera (alpha=1 mantem todos os pixels; alpha=0 corta as bordas pretas).
newK, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 1, (w, h))

# Metodo 1: correcao direta.
undistorted = cv2.undistort(img, K, dist, None, newK)

# Metodo 2: remapping (mapas calculados uma vez, aplicados com remap).
mapx, mapy = cv2.initUndistortRectifyMap(K, dist, None, newK, (w, h), 5)
remapped = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)

# Salva um comparativo lado a lado (original | corrigida).
cv2.imwrite('lab4_undistort_demo.png', cv2.hconcat([img, undistorted]))
cv2.imwrite('lab4_remap_demo.png', remapped)
print("Nova matriz K refinada:\n", newK)
print("ROI valida (x, y, w, h):", roi)


**Resultado obtido pela equipe.** A figura abaixo compara a imagem **original** (esquerda, com a
distorção da lente) com a imagem **corrigida** por `cv2.undistort` (direita), usando os parâmetros do
item (B). Observe, sobretudo perto das bordas, que as linhas do tabuleiro e a borda azul da placa, antes
levemente **curvadas** pelo efeito barril da lente, ficam **retas** após a correção; nas laterais surgem
pequenas regiões pretas, onde não há mais informação de pixel válida (a `roi` retornada indica a área
aproveitável).

![Correção de distorção: original à esquerda, corrigida com cv.undistort à direita](lab4_undistort_demo.png)

*Figura 1 — Saída do programa de correção do item (D): à esquerda a imagem `roberto3.jpg` original; à
direita a mesma imagem após `cv2.undistort` com os parâmetros de `parametros_2b.txt`.*

O método de *remapping* (`cv2.remap`) produz um resultado visualmente idêntico ao de `cv2.undistort` —
como esperado, pois ambos aplicam o mesmo modelo de distorção; a diferença é apenas de eficiência quando
se corrigem várias imagens.

![Correção de distorção pelo método de remapping (cv.remap)](lab4_remap_demo.png)

*Figura 2 — Mesma correção pelo método `cv2.initUndistortRectifyMap` + `cv2.remap`.*

> **Nota para completar (item D).** O roteiro pede **pelo menos duas imagens por integrante**, sendo uma
> da webcam e outra de uma câmera diferente. A demonstração acima usa uma imagem real da equipe e os
> parâmetros reais da calibração; para a versão final, basta reexecutar a célula trocando o nome do
> arquivo de entrada (e os parâmetros, no caso da segunda câmera do item C) para gerar os demais pares
> original/corrigida de cada integrante.


## 5. Análise e discussão

**Qualidade das calibrações.** As duas calibrações convergiram para matrizes $K$ fisicamente
plausíveis: *skew* nulo, razão de aspecto $\approx 1$ e ponto principal próximo do centro da imagem em
ambos os casos. A principal diferença entre a câmera de exemplo (item A, $f \approx 536$ px) e a da
equipe (item B, $f \approx 692$ px) está na **distância focal em pixels** e, portanto, no **campo de
visão**: para o mesmo sensor $640\times480$, a câmera da equipe "enxerga" um ângulo mais estreito.

**Intrínsecos × extrínsecos.** O experimento deixou claro, na prática, o papel de cada grupo de
parâmetros. Os **intrínsecos** ($K$, `dist`) são propriedades fixas da câmera e apareceram **uma única
vez** por calibração; os **extrínsecos** ($R$, $t$) mudaram **a cada imagem**, pois descrevem a pose do
tabuleiro relativa à câmera em cada captura. É essa separação que permite, depois, usar a mesma câmera
calibrada para medir cenas completamente diferentes.

**Correção de distorção.** No item (D), a comparação lado a lado confirmou visualmente o efeito dos
coeficientes `dist`: as linhas retas do tabuleiro, encurvadas pela lente, voltaram a ficar retas após a
correção. Isso valida numericamente a calibração — se os coeficientes estivessem errados, a correção
introduziria distorção em vez de removê-la.

**Boas práticas e cuidados observados.** (i) O `CHECKERBOARD` deve contar **cantos internos** (6×8), não
quadrados; um valor errado faz `findChessboardCorners` falhar silenciosamente. (ii) O refino subpixel
(`cornerSubPix`) melhora sensivelmente a precisão. (iii) Deve-se evitar **misturar imagens de câmeras
diferentes** numa mesma calibração — daí a recomendação, no item (C), de conferir os padrões de `glob`.
(iv) Convém registrar sempre o **erro de reprojeção (RMS)** como métrica objetiva de qualidade.

**Pendências desta versão.** Ficam a cargo da equipe: (1) executar e reportar os parâmetros da **segunda
câmera** (item C) a partir de `roberto_outra_camera*.jpg`, com a comparação frente ao item (B); e (2)
gerar os pares original/corrigida **por integrante** exigidos no item (D). As células correspondentes já
estão prontas neste notebook.


## 6. Declaração de uso de Inteligência Artificial Generativa

Em atendimento à **Portaria CNPq nº 2664/2026** (diretrizes de integridade na pesquisa quanto ao uso de
Inteligência Artificial Generativa — IAG), a equipe declara, de forma transparente:

- **Ferramenta utilizada:** assistente de IA generativa (modelo de linguagem da Anthropic — Claude).
- **Finalidade:** apoio à **redação e organização textual** deste relatório (estruturação das seções,
  revisão de texto, comentários do código e formatação do notebook) e à **explicação didática** dos
  conceitos teóricos (modelo pinhole, intrínsecos/extrínsecos, distorção de lentes).
- **Dados experimentais:** todos os parâmetros de calibração ($K$, `dist`, $R$, $t$), as imagens e os
  resultados foram **obtidos pela própria equipe** nos experimentos; a IAG **não gerou dados
  experimentais**. A correção de distorção do item (D) foi computada sobre as imagens e parâmetros reais
  da equipe.
- **Responsabilidade:** a equipe é integralmente responsável pelo conteúdo final, tendo revisado e
  validado todas as informações aqui apresentadas.


## 7. Referências

1. LearnOpenCV — *Geometry of Image Formation*. Disponível em:
   https://learnopencv.com/geometry-of-image-formation/
2. LearnOpenCV — *Camera Calibration using OpenCV*. Disponível em:
   https://learnopencv.com/camera-calibration-using-opencv/
3. OpenCV — *Camera Calibration* (distorção radial das lentes). Disponível em:
   https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html
4. *Camera Calibration Toolbox for Matlab*. Disponível em:
   http://robots.stanford.edu/cs223b04/JeanYvesCalib/
5. *Pinhole camera model* — Wikipedia. Disponível em:
   https://en.wikipedia.org/wiki/Pinhole_camera_model
